# Yield X 데이터 탐색 (C)

이 노트북은 `make data`가 만든 데이터를 읽어 품질과 분포를 탐색하는 **읽기 전용 검토 화면**입니다. 생성·스키마·전처리 로직을 복사하지 않고 production 모듈을 호출하며, 어떤 artifact도 저장하지 않습니다. 먼저 저장소 루트에서 `make data`를 실행하세요.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("저장소 루트 또는 notebooks/ 에서 실행하세요.")
SRC = str(ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd

from yeda.data.preprocess import load_raw
from yeda.io_utils import load_config, resolve
from yeda.schema import FEATURE_NAMES, FEATURES, TARGET, validate_frame

In [ ]:
df = load_raw()
contract = validate_frame(df)
assert contract.ok, contract.errors
df.head()

In [ ]:
data_cfg = load_config("data_gen")
report_path = resolve("artifacts/metrics/data_generation.json")
generation_report = json.loads(report_path.read_text(encoding="utf-8"))
target_check = pd.DataFrame(data_cfg["targets"], index=["target_low", "target_high"]).T
target_check["observed"] = [
    generation_report["success_rate"],
    generation_report["bayes_accuracy"],
]
target_check

In [ ]:
schema_table = pd.DataFrame(
    [
        {
            "feature": spec.name,
            "unit": spec.unit,
            "kind": spec.kind,
            "low": spec.low,
            "high": spec.high,
            "resolution": spec.resolution,
            "adjustable": spec.adjustable,
        }
        for spec in FEATURES
    ]
)
schema_table

In [ ]:
quality = df[list(FEATURE_NAMES)].agg(["min", "median", "max"]).T
quality["missing_rate"] = df[list(FEATURE_NAMES)].isna().mean()
quality

In [ ]:
target_distribution = df[TARGET].value_counts(dropna=False).sort_index().to_frame("count")
target_distribution["rate"] = target_distribution["count"] / len(df)
target_distribution